In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("Evaluation finale")
         .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.12.2")
         .getOrCreate())
spark.sparkContext.setCheckpointDir("checkpoints")

In [2]:
abonnements = spark.read.csv("abonnements.csv", header=True, inferSchema=True)
personnes = spark.read.csv("personnes.csv", header=True, inferSchema=True)

In [3]:
abonnements.show()

+---+---+
|src|dst|
+---+---+
|  b|  a|
|  c|  a|
|  d|  a|
|  e|  a|
|  f|  a|
|  g|  a|
|  h|  a|
|  a|  b|
|  a|  c|
|  c|  d|
|  d|  c|
|  e|  f|
|  f|  e|
|  g|  h|
|  h|  i|
|  i|  j|
|  j|  g|
|  b|  e|
|  d|  g|
|  k|  i|
+---+---+
only showing top 20 rows


In [4]:
abonnements.count()

24

In [5]:
personnes.show()

+---+-----+
| id|  nom|
+---+-----+
|  a|  Awa|
|  b|Bruno|
|  c|Chloé|
|  d|David|
|  e| Emma|
|  f|Farid|
|  g|Grace|
|  h| Hugo|
|  i| Inès|
|  j|Jules|
|  k|Karim|
|  l|  Léa|
|  m|Malik|
|  n| Nina|
+---+-----+



In [6]:
personnes.count()

14

In [8]:
from graphframes import GraphFrame

In [9]:
reseau = GraphFrame(personnes, abonnements)

In [10]:
print(reseau.vertices.count(), "sommets,", reseau.edges.count(), "arêtes")

14 sommets, 24 arêtes


In [14]:
reseau.inDegrees.join(personnes, "id").orderBy(F.desc("inDegree"), "nom").show(3)

+---+--------+-----+
| id|inDegree|  nom|
+---+--------+-----+
|  a|       7|  Awa|
|  c|       2|Chloé|
|  e|       2| Emma|
+---+--------+-----+
only showing top 3 rows


In [16]:
mutus = (reseau.find("(x)-[e1]->(y); (y)-[e2]->(x)"))

In [18]:
mutus.count()

8

In [23]:
mutus.select(F.col("x.nom").alias("personne1"), F.col("y.nom").alias("personne2"), F.col("x.id"), F.col("y.id")).show()

+---------+---------+---+---+
|personne1|personne2| id| id|
+---------+---------+---+---+
|    Bruno|      Awa|  b|  a|
|    Chloé|      Awa|  c|  a|
|      Awa|    Bruno|  a|  b|
|      Awa|    Chloé|  a|  c|
|    Chloé|    David|  c|  d|
|    David|    Chloé|  d|  c|
|     Emma|    Farid|  e|  f|
|    Farid|     Emma|  f|  e|
+---------+---------+---+---+



In [21]:
mutus.filter("x.id < y.id").select(F.col("x.nom").alias("personne1"), F.col("y.nom").alias("personne2")).show()

+---------+---------+
|personne1|personne2|
+---------+---------+
|      Awa|    Bruno|
|      Awa|    Chloé|
|    Chloé|    David|
|     Emma|    Farid|
+---------+---------+



In [24]:
composantes = reseau.connectedComponents()
(composantes.groupBy("component")
            .agg(F.count("*").alias("nb_personnes"), F.collect_list("nom").alias("personnes"))
            .orderBy(F.desc("nb_personnes")).show(truncate=False))

+------------+------------+------------------------------------------------------------------------+
|component   |nb_personnes|personnes                                                               |
+------------+------------+------------------------------------------------------------------------+
|146028888064|11          |[Awa, Bruno, Chloé, David, Emma, Farid, Grace, Hugo, Inès, Jules, Karim]|
|51539607552 |3           |[Léa, Malik, Nina]                                                      |
+------------+------------+------------------------------------------------------------------------+



In [25]:
chemins = reseau.bfs(fromExpr="nom = 'Karim'", toExpr="nom = 'Awa'")

In [26]:
chemins.show()

+----------+------+----------+------+----------+------+--------+
|      from|    e0|        v1|    e1|        v2|    e2|      to|
+----------+------+----------+------+----------+------+--------+
|{k, Karim}|{k, j}|{j, Jules}|{j, g}|{g, Grace}|{g, a}|{a, Awa}|
+----------+------+----------+------+----------+------+--------+



In [28]:
chemins.select(F.col("from.nom").alias("départ"), F.col("v1.nom").alias("int_1"),
               F.col("v2.nom").alias("int_2"),
               F.col("to.nom").alias("arrivee")).show()

+------+-----+-----+-------+
|départ|int_1|int_2|arrivee|
+------+-----+-----+-------+
| Karim|Jules|Grace|    Awa|
+------+-----+-----+-------+



In [29]:
influence = reseau.pageRank(resetProbability=0.15, maxIter=10)

In [31]:
influence.vertices.show()

+---+-----+-------------------+
| id|  nom|           pagerank|
+---+-----+-------------------+
|  f|Farid| 0.5708463082153786|
|  g|Grace| 0.9002081652556351|
|  k|Karim|0.15000000000000002|
|  m|Malik| 1.0000000000000002|
|  a|  Awa| 2.9447634529537923|
|  j|Jules| 0.5910355247437256|
|  e| Emma| 0.9853939941488612|
|  d|David| 0.8467562169264753|
|  n| Nina| 1.0000000000000002|
|  i| Inès|0.43801681661748204|
|  h| Hugo| 0.5266770636941656|
|  b|Bruno| 1.4027397020465264|
|  l|  Léa| 1.0000000000000002|
|  c|Chloé| 1.6435627553979566|
+---+-----+-------------------+



In [32]:
(influence.vertices.select("nom", F.round("pagerank", 2).alias("pagerank"))
           .orderBy(F.desc("pagerank"), "nom").show())

+-----+--------+
|  nom|pagerank|
+-----+--------+
|  Awa|    2.94|
|Chloé|    1.64|
|Bruno|     1.4|
|  Léa|     1.0|
|Malik|     1.0|
| Nina|     1.0|
| Emma|    0.99|
|Grace|     0.9|
|David|    0.85|
|Jules|    0.59|
|Farid|    0.57|
| Hugo|    0.53|
| Inès|    0.44|
|Karim|    0.15|
+-----+--------+



In [33]:
(influence.vertices.select("nom", F.round("pagerank", 2).alias("pagerank"))
           .orderBy(F.desc("pagerank"), "nom").show(3))

+-----+--------+
|  nom|pagerank|
+-----+--------+
|  Awa|    2.94|
|Chloé|    1.64|
|Bruno|     1.4|
+-----+--------+
only showing top 3 rows


In [34]:
resultat_influence = (influence.vertices.select("nom", F.round("pagerank", 2).alias("pagerank"))
           .orderBy(F.desc("pagerank"), "nom"))

In [35]:
resultat_influence.write.mode("overwrite").parquet("resultats/j3/influence")

In [36]:
relu = spark.read.parquet("resultats/j3/influence")

In [37]:
relu.count()

14